# 🎓 FaceTrack AI — Google Colab Testing

Run the **full** Face Recognition Attendance System on Google Colab with **NVIDIA GPU acceleration** and the **complete web UI** — student management, live recognition, attendance reports, settings.

---

## ⚠️ Before you start — Set runtime to T4 GPU

> `Runtime` → `Change runtime type` → `Hardware accelerator` → **T4 GPU** → Save

---

## 🚀 One-click start

Press **`Ctrl + F9`** (or `Runtime → Run all`) to run every cell automatically.  
The **Cleanup cell at the bottom is safely guarded** — it won't stop anything unless you manually set `_AUTO_STOP = True` inside it.

---

## How it works

| Component | Local PC | Google Colab |
|---|---|---|
| **Camera** | USB webcam via `cv2.VideoCapture` | Browser webcam via JS → POSTs frames to Flask |
| **Inference** | OpenVINO CPU EP | `onnxruntime-gpu` → CUDA EP (NVIDIA T4) |
| **Web UI** | `http://localhost:5000` | Public HTTPS URL via **Cloudflare Tunnel** (no account needed) |
| **Frontend** | Pre-built `frontend/dist/` | Built in this notebook after `git clone` |

In [ ]:
# ✅ Cell 1 — Verify GPU is connected
# ─────────────────────────────────────────────────────────────────────────────
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected:\n")
    print(result.stdout)
else:
    print("❌  NO GPU DETECTED!")
    print("─" * 60)
    print("Go to:  Runtime  ›  Change runtime type")
    print("Set 'Hardware accelerator' to  T4 GPU  then save.")
    print("Then:  Runtime  ›  Restart session  and re-run all cells.")
    print("─" * 60)
    raise SystemExit("Enable GPU before continuing.")

In [ ]:
%%bash
# 📦 Cell 2 — Install Colab-specific dependencies
# onnxruntime-gpu (CUDA) replaces onnxruntime-openvino (Intel-only)
# opencv-contrib-python-headless: no GUI deps (no display server on Colab VM)
set -e

echo "► Installing system libraries..."
apt-get install -qq libgl1 libglib2.0-0 2>/dev/null | tail -1

echo "► Installing cloudflared (zero-setup HTTPS tunnel — no account needed)..."
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
     -O /tmp/cloudflared.deb
dpkg -i /tmp/cloudflared.deb | tail -2

echo "► Installing Python packages (may take 2-3 minutes)..."
pip install -q \
    insightface \
    onnxruntime-gpu \
    onnx \
    "opencv-contrib-python-headless>=4.9.0" \
    "scikit-learn>=1.4.0" \
    "flask>=3.0.0" \
    "flask-socketio>=5.3.6" \
    "waitress>=3.0.0" \
    "python-dotenv>=1.0.1" \
    "flask-limiter>=3.5.0" \
    "Werkzeug>=3.0.1" \
    "reportlab>=4.1.0" \
    "openpyxl>=3.1.2" \
    "pandas>=2.2.0" \
    "Pillow>=10.2.0" \
    "numpy>=1.26.0"

echo ""
echo "✅ All dependencies installed."

In [ ]:
# 📂 Cell 3 — Clone repository and build the Vue frontend
# ─────────────────────────────────────────────────────────────────────────────
# frontend/dist/ is gitignored and must be built here after cloning.
# Colab already has Node.js installed, so this just runs npm install + build.
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess

REPO_URL = "https://github.com/Abhirup-1234/Face-Recognition-Attendance-System.git"
REPO_DIR = "/content/FaceTrack"

# ── Clone or update repo ─────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Repository cloned.")
else:
    print("Repository already exists — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✅ Repository up to date.")

# ── Create .env with test credentials ────────────────────────────────────────
# config.py loads this on import via python-dotenv
with open(f"{REPO_DIR}/.env", "w") as f:
    f.write(
        "SECRET_KEY=colab-test-secret-key-9f4a2b8c3d1e\n"
        "ADMIN_USERNAME=admin\n"
        "ADMIN_PASSWORD=admin123\n"
    )
print("\nCredentials: admin / admin123")

# ── Build the Vue frontend ────────────────────────────────────────────────────
# frontend/dist/ is gitignored. Flask's _serve_spa() needs index.html there.
FRONTEND_DIR = f"{REPO_DIR}/frontend"
DIST_INDEX   = f"{FRONTEND_DIR}/dist/index.html"

if not os.path.exists(DIST_INDEX):
    print("\n🔨 Building Vue frontend (npm install + npm run build)...")
    print("   This takes about 1-2 minutes on first run.\n")
    subprocess.run(["npm", "install"], cwd=FRONTEND_DIR, check=True)
    subprocess.run(["npm", "run", "build"], cwd=FRONTEND_DIR, check=True)
    print("\n✅ Frontend built successfully.")
else:
    print("\n✅ Frontend already built — skipping.")

print(f"\nRepository : {REPO_DIR}")
print("✅ Ready. Run the next cell.")

In [ ]:
# ⚙️ Cell 4 — Configure for Colab and start Flask
# ─────────────────────────────────────────────────────────────────────────────
# What this cell does:
#   1. Patches config BEFORE importing app:
#      - CAMERAS      → "push://CAM-COLAB" (PushCameraProcessor, no VideoCapture)
#      - CORS_ORIGINS → "*" (required for Cloudflare tunnel + Colab iframe)
#   2. Patches FaceEngine to use CUDAExecutionProvider instead of OpenVINO
#   3. Creates the Flask app, adds CORS hooks, starts server in background
# ─────────────────────────────────────────────────────────────────────────────
import sys, os, threading, time, webbrowser
from pathlib import Path

REPO_DIR = "/content/FaceTrack"

# Add src/ to path so all imports work exactly as in local development
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.chdir(REPO_DIR)

# Suppress webbrowser.open() — Colab VM has no GUI
webbrowser.open = lambda url, **kw: None

# ── Patch config BEFORE importing app ────────────────────────────────────────
# SocketIO reads config.CORS_ORIGINS at module-import time, so we must
# override these values before `import app`.
import config

# Push camera: PushCameraProcessor activates when source startswith "push://".
# The local PC config ("CAM-101": 1) is never touched.
config.CAMERAS = {"CAM-COLAB": "push://CAM-COLAB"}

# Allow all origins so the Cloudflare URL and Colab notebook iframe
# can reach Flask routes (SocketIO handshake + push_frame POST).
config.CORS_ORIGINS = "*"

print(f"BASE_DIR : {config.BASE_DIR}")
print(f"DATA_DIR : {config.DATA_DIR}")
print(f"CAMERAS  : {config.CAMERAS}")

# ── Import app (SocketIO picks up the patched CORS_ORIGINS at import time) ───
print("\n🔄 Loading app... (InsightFace buffalo_l models ~500 MB on first run)")
import app as app_module
from face_engine import FaceEngine
import onnxruntime as ort

# ── Verify CUDA and patch FaceEngine provider selection ──────────────────────
eps = ort.get_available_providers()
print(f"\nONNX Runtime {ort.__version__}")
print(f"Available providers: {eps}")

if "CUDAExecutionProvider" in eps:
    print("✅ CUDAExecutionProvider available — NVIDIA GPU will be used!")
else:
    print("⚠️  CUDAExecutionProvider not found — will fall back to CPU.")
    print("   Make sure you selected T4 GPU in Runtime settings.")


def _colab_select_providers():
    """Drop-in replacement for FaceEngine._select_providers() in Colab.
    Uses CUDA instead of OpenVINO (which requires Intel hardware)."""
    available = ort.get_available_providers()
    if "CUDAExecutionProvider" in available:
        print("[FaceEngine] ✅ Using CUDAExecutionProvider (NVIDIA GPU)")
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    print("[FaceEngine] ⚠️  CUDA unavailable — using CPUExecutionProvider")
    return ["CPUExecutionProvider"]


# Patch before any FaceEngine instance is created
FaceEngine._select_providers = staticmethod(_colab_select_providers)
FaceEngine._instance = None   # reset singleton so __init__ re-runs with CUDA

# ── Create Flask app and attach CORS headers ─────────────────────────────────
# CORS is needed so the Colab notebook iframe (*.googleusercontent.com)
# can POST frames to the push_frame endpoint via the Cloudflare tunnel URL.
flask_app = app_module.create_app()

from flask import request as _flask_req


@flask_app.before_request
def _handle_preflight():
    """Return CORS pre-flight response for all OPTIONS requests."""
    if _flask_req.method == "OPTIONS":
        r = flask_app.make_default_options_response()
        r.headers["Access-Control-Allow-Origin"]  = "*"
        r.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS, DELETE, PUT"
        r.headers["Access-Control-Allow-Headers"] = "Content-Type"
        return r


@flask_app.after_request
def _add_cors(response):
    """Attach CORS headers to every response."""
    response.headers["Access-Control-Allow-Origin"]  = "*"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS, DELETE, PUT"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type"
    return response


# ── Start Flask in a background thread ───────────────────────────────────────
_server_ready = threading.Event()


def _run_server():
    # startup() initialises DB, loads face embeddings, starts cameras
    app_module.startup(flask_app)
    _server_ready.set()
    app_module.socketio.run(
        flask_app,
        host="0.0.0.0",
        port=5000,
        use_reloader=False,
        log_output=False,
        allow_unsafe_werkzeug=True,
    )


threading.Thread(target=_run_server, daemon=True).start()

print("\n⏳ Waiting for Flask to initialise (model download can take a few minutes)...")
if _server_ready.wait(timeout=300):
    print("\n✅ Flask server is running on http://localhost:5000")
    print("\n▶ Run the next cell to start the Cloudflare Tunnel.")
else:
    print("❌ Flask didn't start within 5 minutes. Check error output above.")

In [ ]:
# 🌐 Cell 5 — Start Cloudflare Tunnel
# No account, no token, no sign-up. Powered by Cloudflare's global CDN.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, threading, re, time

TUNNEL_URL   = None
_tunnel_proc = None


def _run_tunnel():
    global TUNNEL_URL, _tunnel_proc
    _tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in _tunnel_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
            break
    # Drain remaining output to keep the tunnel process alive
    for _ in _tunnel_proc.stdout:
        pass


threading.Thread(target=_run_tunnel, daemon=True).start()

print("⏳ Starting Cloudflare Tunnel (usually takes 5-15 seconds)...")
for _ in range(45):
    if TUNNEL_URL:
        break
    time.sleep(1)

if TUNNEL_URL:
    print(f"\n{'=' * 64}")
    print(f"  ✅  FaceTrack AI is LIVE on Colab GPU!")
    print(f"{'=' * 64}")
    print(f"\n  🌐  Open this URL in your browser (any device, any network):")
    print(f"      {TUNNEL_URL}")
    print(f"\n  🔑  Login:  admin  /  admin123")
    print(f"\n  📷  Run the NEXT CELL to start streaming your browser webcam.")
    print(f"{'=' * 64}")
else:
    print("❌ Timed out waiting for the tunnel URL.")
    print("   Make sure Flask started (Cell 4 completed successfully).")
    print("   Try running this cell again.")

In [ ]:
# 📷 Cell 6 — Browser Webcam → FaceTrack AI (10 fps)
# ─────────────────────────────────────────────────────────────────────────────
# Embeds a JS widget that:
#   1. Opens your browser webcam (camera permission prompt will appear)
#   2. Captures frames at 10 fps
#   3. POSTs each JPEG to Flask → PushCameraProcessor → FaceEngine (Colab GPU)
#
# The full web UI is at TUNNEL_URL — log in and go to the Camera tab.
# Enable recognition there to start face detection.
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import HTML, display
import config as _cfg

if not TUNNEL_URL:
    print("❌  TUNNEL_URL is not set. Run Cell 5 first.")
else:
    CAMERA_ID = "CAM-COLAB"
    PUSH_URL  = TUNNEL_URL + "/api/push_frame/" + CAMERA_ID

    html = (
        '<div style="font-family:system-ui,sans-serif;padding:14px;'
        'background:#f0f4f8;border-radius:12px;max-width:410px;'
        'box-shadow:0 2px 8px rgba(0,0,0,0.12)">'
        '<h3 style="margin:0 0 10px;color:#1a237e">'
        '&#128247; FaceTrack AI &mdash; Browser Webcam</h3>'
        '<p id="ft-status" style="margin:4px 0;font-size:13px;color:#555">'
        'Requesting camera permission&hellip;</p>'
        '<video id="ft-video" width="370" height="278" autoplay muted playsinline '
        'style="border:3px solid #4CAF50;border-radius:10px;'
        'display:block;margin:10px 0;background:#111">'
        '</video>'
        '<p style="font-size:12px;margin:6px 0;color:#444">'
        'View live annotated feed &amp; full UI at:</p>'
        '<a href="' + TUNNEL_URL + '" target="_blank" '
        'style="font-size:13px;color:#1565C0;word-break:break-all">'
        + TUNNEL_URL + '</a><br><br>'
        '<button id="ft-stop-btn" '
        'onclick="window._ft_stop=true;'
        "document.getElementById('ft-status').textContent='\u23f9 Camera stopped.';" 
        "document.getElementById('ft-stop-btn').disabled=true;" '" '
        'style="padding:6px 18px;border-radius:6px;cursor:pointer;'
        'background:#e53935;color:#fff;border:none;font-size:13px">'
        '&#9209; Stop Camera</button>'
        '</div>'
        '<script>'
        '(async function(){'
        'const v=document.getElementById("ft-video");'
        'const s=document.getElementById("ft-status");'
        'const PUSH="' + PUSH_URL + '";'
        'try{'
        'const stream=await navigator.mediaDevices.getUserMedia('
        '{video:{width:640,height:480,facingMode:"user"}});'
        'v.srcObject=stream;'
        's.textContent="\u2705 Camera active \u2014 streaming to Flask at 10\u2009fps";'
        's.style.color="#2E7D32";'
        'const c=document.createElement("canvas");'
        'c.width=640;c.height=480;'
        'const ctx=c.getContext("2d");'
        'window._ft_stop=false;'
        '(function send(){'
        'if(window._ft_stop)return;'
        'ctx.drawImage(v,0,0,640,480);'
        'c.toBlob(b=>'
        'fetch(PUSH,{method:"POST",'
        'headers:{"Content-Type":"application/octet-stream"},'
        'body:b,mode:"cors"})'
        '.catch(e=>console.warn("[FaceTrack] push error:",e))'
        ',"image/jpeg",0.8);'
        'setTimeout(send,100);})();'
        '}catch(e){'
        's.textContent="\u274c Camera access denied: "+e.message;'
        's.style.color="#b71c1c";}'
        '})();'
        '</script>'
    )

    display(HTML(html))

    print(f"\n✅ Webcam widget started.")
    print(f"\n📱 Open the web UI:  {TUNNEL_URL}")
    print(f"   Login: admin / admin123")
    print(f"   Camera tab → Enable Recognition to start GPU face detection")
    print(f"   Attendance auto-marks after {_cfg.CONFIRM_FRAMES} consecutive detections")
    print(f"\n   Enroll students via the Enroll tab in the web UI.")

---
## 🛑 Cleanup — run only when you want to stop

The cell below **does nothing** by default, so `Runtime → Run all` is safe.  
When you're done testing, change `_AUTO_STOP = False` → `True` and run just this cell.

In [ ]:
# 🛑 Cleanup — change _AUTO_STOP to True and run this cell to stop
# ─────────────────────────────────────────────────────────────────────────────
# Keeping _AUTO_STOP = False means Runtime → Run all skips this safely.
# ─────────────────────────────────────────────────────────────────────────────
_AUTO_STOP = False   # ← change to True when you want to stop

if _AUTO_STOP:
    print("Stopping Cloudflare Tunnel...")
    try:
        _tunnel_proc.terminate()
        print("✅ Tunnel stopped.")
    except Exception as e:
        print(f"⚠️  {e}")
    print("\nFlask server and GPU threads stop when you disconnect the runtime.")
    print("Go to:  Runtime  ›  Disconnect and delete runtime")
else:
    print("ℹ️  Cleanup skipped (_AUTO_STOP is False).")
    print("   Set _AUTO_STOP = True and re-run this cell to stop the tunnel.")